In [1]:
# Requirements
# Install the following packages before running this notebook:
# 
#   pip install numpy plotly ipywidgets pyvista vtk
#
# Or use: pip install -r requirements.txt (contents below)
# 
# numpy
# plotly
# ipywidgets
# pyvista
# vtk


In [2]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
import os

# Prefer pyvista (simpler API), fall back to raw vtk if not installed
try:
    import pyvista as pv
    from vtk.util.numpy_support import vtk_to_numpy
    HAS_PYVISTA = True
except ImportError:
    HAS_PYVISTA = False
    import vtk
    from vtk.util.numpy_support import vtk_to_numpy

# Look for a .vti file in the current directory; ask for one if none found
vti_filename = None
for f in os.listdir('.'):
    if f.endswith('.vti'):
        vti_filename = f
        break

if vti_filename is None:
    vti_filename = input("Enter the VTI filename: ")

if HAS_PYVISTA:
    mesh = pv.read(vti_filename)
    dims = mesh.dimensions
    bounds = mesh.bounds
    vtk_point_data = mesh.GetPointData()
    vtk_scalars = vtk_point_data.GetScalars()
    if vtk_scalars is None:
        # No active scalars set, fall back to the first available array
        if vtk_point_data.GetNumberOfArrays() > 0:
            vtk_scalars = vtk_point_data.GetArray(0)
        else:
            raise ValueError("No scalar data found.")
    scalar_data = vtk_to_numpy(vtk_scalars)
else:
    # Raw VTK path: read the .vti image data directly
    reader = vtk.vtkXMLImageDataReader()
    reader.SetFileName(vti_filename)
    reader.Update()
    image_data = reader.GetOutput()
    dims = image_data.GetDimensions()
    bounds = image_data.GetBounds()
    vtk_scalars = image_data.GetPointData().GetScalars()
    if vtk_scalars is None:
        raise ValueError("No scalar data found.")
    scalar_data = vtk_to_numpy(vtk_scalars)

# Ensure consistent dtype/shape regardless of which backend produced the array
scalar_data = np.asarray(scalar_data, dtype=np.float64).ravel()

# Build the regular grid of point coordinates from the volume dimensions/bounds
nx, ny, nz = dims[0], dims[1], dims[2]
x_coords = np.linspace(bounds[0], bounds[1], nx)
y_coords = np.linspace(bounds[2], bounds[3], ny)
z_coords = np.linspace(bounds[4], bounds[5], nz)

X, Y, Z = np.meshgrid(x_coords, y_coords, z_coords, indexing='ij')

# Flatten in Fortran order to match VTK's point ordering (x fastest-varying)
x_flat = X.flatten(order='F')
y_flat = Y.flatten(order='F')
z_flat = Z.flatten(order='F')
values_flat = scalar_data

# Cache global min/max once - reused for color scaling and slider bounds
DATA_MIN = float(values_flat.min())
DATA_MAX = float(values_flat.max())

In [3]:
# Default isovalue the visualization starts (and resets) at
INITIAL_ISOVALUE = 0.0
full_hist_data = values_flat.copy()

# Isosurface plot: isomin == isomax with surface_count=1 renders a single iso-level surface
fig_isosurface = go.FigureWidget(data=[go.Isosurface(
    x=x_flat,
    y=y_flat,
    z=z_flat,
    value=values_flat,
    isomin=INITIAL_ISOVALUE,
    isomax=INITIAL_ISOVALUE,
    surface_count=1,
    colorscale='plasma',
    cmin=DATA_MIN,
    cmax=DATA_MAX,
    showscale=False,
    opacity=0.9,
    hoverinfo='skip'
)])

fig_isosurface.update_layout(
    title=dict(text='Isosurface Visualization', x=0.5),
    scene=dict(
        xaxis=dict(title='x', showbackground=True, backgroundcolor='rgb(230, 230, 250)'),
        yaxis=dict(title='y', showbackground=True, backgroundcolor='rgb(230, 230, 250)'),
        zaxis=dict(title='z', showbackground=True, backgroundcolor='rgb(230, 230, 250)'),
        aspectmode='data',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
    ),
    width=700,
    height=600,
    margin=dict(l=20, r=20, t=50, b=20),
    paper_bgcolor='white',
    plot_bgcolor='white'
)

# Histogram starts out showing the full volume's scalar distribution
fig_histogram = go.FigureWidget(data=[go.Histogram(
    x=full_hist_data,
    nbinsx=30,
    marker=dict(
        color='rgba(0, 0, 255, 0.5)',
        line=dict(color='darkblue', width=1)
    ),
    opacity=0.7
)])

fig_histogram.update_layout(
    title=dict(text='Histogram of Entire Volume', x=0.5),
    xaxis=dict(
        title='Vortex scalar values',
        range=[DATA_MIN, DATA_MAX]
    ),
    yaxis=dict(
        title='Frequency',
        rangemode='nonnegative'
    ),
    width=600,
    height=600,
    margin=dict(l=60, r=20, t=50, b=50),
    paper_bgcolor='white',
    plot_bgcolor='rgb(240, 240, 255)'
)

# Slider drives the isovalue; step size gives 200 increments across the data range
slider = widgets.FloatSlider(
    value=INITIAL_ISOVALUE,
    min=DATA_MIN,
    max=DATA_MAX,
    step=(DATA_MAX - DATA_MIN) / 200,
    description='Isoval:',
    continuous_update=False,
    readout=True,
    readout_format='.2f',
    layout=widgets.Layout(width='500px')
)

# Button to restore the initial isovalue/view
reset_button = widgets.Button(
    description='Reset',
    button_style='info',
    tooltip='Reset to initial state',
    layout=widgets.Layout(width='120px', height='35px')
)

controls = widgets.HBox(
    [slider, reset_button],
    layout=widgets.Layout(justify_content='flex-start', margin='10px 0px')
)

plots = widgets.HBox(
    [fig_isosurface, fig_histogram],
    layout=widgets.Layout(justify_content='center')
)

# Final layout: controls on top, the two plots side by side below
ui = widgets.VBox([controls, plots])

In [4]:
# Called whenever the slider value changes
def update_visualization(change=None):
    isoval = slider.value

    # Move the isosurface to the new isovalue
    with fig_isosurface.batch_update():
        fig_isosurface.data[0].isomin = isoval
        fig_isosurface.data[0].isomax = isoval

    # Restrict the histogram to a small band around the current isovalue
    lower_bound = isoval - 0.25
    upper_bound = isoval + 0.25

    subset_mask = (values_flat >= lower_bound) & (values_flat <= upper_bound)
    subset_data = values_flat[subset_mask]

    if subset_data.size > 0:
        with fig_histogram.batch_update():
            fig_histogram.data[0].x = subset_data
            fig_histogram.data[0].nbinsx = 30
            fig_histogram.update_layout(
                title=dict(text=f'Histogram (subset: {lower_bound:.2f} to {upper_bound:.2f})', x=0.5),
                xaxis=dict(
                    title='Vortex scalar values',
                    range=[lower_bound, upper_bound]
                ),
                yaxis=dict(
                    title='Frequency',
                    rangemode='nonnegative'
                )
            )
    else:
        # No points fall in this band - show an empty histogram instead of erroring
        with fig_histogram.batch_update():
            fig_histogram.data[0].x = []
            fig_histogram.update_layout(
                title=dict(text='Histogram (no data in range)', x=0.5),
                xaxis=dict(range=[lower_bound, upper_bound])
            )


# Called when the Reset button is clicked
def reset_visualization(button=None):
    # Restore the slider, isosurface, and histogram to their initial state
    slider.value = INITIAL_ISOVALUE

    with fig_isosurface.batch_update():
        fig_isosurface.data[0].isomin = INITIAL_ISOVALUE
        fig_isosurface.data[0].isomax = INITIAL_ISOVALUE

    with fig_histogram.batch_update():
        fig_histogram.data[0].x = full_hist_data
        fig_histogram.data[0].nbinsx = 30
        fig_histogram.update_layout(
            title=dict(text='Histogram of Entire Volume', x=0.5),
            xaxis=dict(
                title='Vortex scalar values',
                range=[DATA_MIN, DATA_MAX]
            ),
            yaxis=dict(
                title='Frequency',
                rangemode='nonnegative'
            )
        )


# Wire up the callbacks and render the UI
slider.observe(update_visualization, names='value')
reset_button.on_click(reset_visualization)

display(ui)